In [ ]:
"""
Glaucoma KG Parser: parquet -> triples -> Neo4j-ready CSV
---------------------------------------------------------
Input : train-00000-of-00001.parquet  +  test-00000-of-00001.parquet
Output: kg_nodes.csv + kg_edges.csv  (Cypher LOAD CSV compatible)
 
Node types : FundusImage, OpticDisc, NeuralRim, Pathology, Diagnosis, RiskLevel
Edge types : HAS_OPTIC_DISC, HAS_RIM, HAS_PATHOLOGY, HAS_DIAGNOSIS,
             HAS_RISK, SUPPORTS_DIAGNOSIS
"""
 
import json
import uuid
import argparse
import pandas as pd
from pathlib import Path
 
 # 无聊的导入包

In [ ]:
DATA_DIR  = Path("Dataset/Data")
OUT_DIR   = Path("kg_output")
OUT_DIR.mkdir(exist_ok=True)
 
train_path = DATA_DIR / "train-00000-of-00001.parquet"
test_path  = DATA_DIR / "test-00000-of-00001.parquet"
 
train_df = pd.read_parquet(train_path)
test_df  = pd.read_parquet(test_path)
train_df["_split"] = "train"
test_df["_split"]  = "test"
 
df = pd.concat([train_df, test_df], ignore_index=True)
print(f"Train: {len(train_df)} rows")
print(f"Test : {len(test_df)} rows")
print(f"Total: {len(df)} rows")
print(f"\nColumns: {list(df.columns)}")

# 首先查看项目的路径

Train: 589 rows
Test : 100 rows
Total: 689 rows

Columns: ['image', 'label', 'filename', 'annotation', 'description', '_split']


In [8]:
sample = df.iloc[0].to_dict()
print("filename  :", sample.get("filename"))
print("annotation:", sample.get("annotation"))
print("label     :", sample.get("label"))
desc = json.loads(sample.get("description", "{}"))
print("\ndescription (parsed):")
print(json.dumps(desc, indent=2, ensure_ascii=False))

# 瞅瞅一个案例

filename  : acrima_glaucoma_57.png
annotation: glaucoma
label     : 0

description (parsed):
{
  "fundus_features": {
    "optic_disc_size": "large",
    "cup_to_disc_ratio": 0.8,
    "neuroretinal_rim": "thinned in all regions, especially inferior and superior",
    "isnt_rule_followed": false,
    "rim_pallor": true,
    "rim_color": "pale",
    "bayoneting": true,
    "sharp_edge": true,
    "laminar_dot_sign": true,
    "notching": true,
    "rim_thinning": true,
    "additional_observations": null
  },
  "glaucoma_risk_assessment": "high risk",
  "confidence_level": 0.9
}


In [9]:
def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:8]}"
 
def safe_json(raw) -> dict:
    if not isinstance(raw, str):
        return {}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}
 
def extract_record(row: dict, split: str = "") -> tuple[list[dict], list[dict]]:
    nodes, edges = [], []
    desc = safe_json(row.get("description", "{}"))
    ff   = desc.get("fundus_features", {})
 
    # FundusImage
    img_id = new_id("img")
    nodes.append({
        "id": img_id, "label": "FundusImage",
        "filename": row.get("filename", ""),
        "annotation": row.get("annotation", ""),
        "bin_label": int(row.get("label", -1)),
        "split": split,
    })
 
    # OpticDisc
    od_id = new_id("od")
    nodes.append({
        "id": od_id, "label": "OpticDisc",
        "optic_disc_size":   ff.get("optic_disc_size"),
        "cup_to_disc_ratio": ff.get("cup_to_disc_ratio"),
        "sharp_edge":        ff.get("sharp_edge"),
    })
    edges.append({"src": img_id, "rel": "HAS_OPTIC_DISC", "dst": od_id})
 
    # NeuralRim
    rim_id = new_id("rim")
    nodes.append({
        "id": rim_id, "label": "NeuralRim",
        "neuroretinal_rim":   ff.get("neuroretinal_rim"),
        "isnt_rule_followed": ff.get("isnt_rule_followed"),
        "rim_pallor":         ff.get("rim_pallor"),
        "rim_color":          ff.get("rim_color"),
        "rim_thinning":       ff.get("rim_thinning"),
    })
    edges.append({"src": img_id, "rel": "HAS_RIM", "dst": rim_id})
 
    # Pathology
    path_id = new_id("path")
    nodes.append({
        "id": path_id, "label": "Pathology",
        "bayoneting":              ff.get("bayoneting"),
        "notching":                ff.get("notching"),
        "laminar_dot_sign":        ff.get("laminar_dot_sign"),
        "additional_observations": ff.get("additional_observations"),
    })
    edges.append({"src": img_id, "rel": "HAS_PATHOLOGY", "dst": path_id})
 
    # Diagnosis
    diag_id  = new_id("diag")
    risk_raw = desc.get("glaucoma_risk_assessment")
    conf     = desc.get("confidence_level")
    nodes.append({
        "id": diag_id, "label": "Diagnosis",
        "annotation":      row.get("annotation", ""),
        "risk_assessment": risk_raw,
        "confidence":      conf,
    })
    edges.append({"src": img_id, "rel": "HAS_DIAGNOSIS", "dst": diag_id})
 
    # RiskLevel
    if risk_raw:
        risk_id = f"risk_{risk_raw.lower().replace(' ', '_')}"
        nodes.append({"id": risk_id, "label": "RiskLevel", "value": risk_raw})
        edges.append({"src": diag_id, "rel": "HAS_RISK", "dst": risk_id})
 
    # SUPPORTS_DIAGNOSIS 推理边
    if ff.get("cup_to_disc_ratio") is not None:
        edges.append({
            "src": od_id, "rel": "SUPPORTS_DIAGNOSIS", "dst": diag_id,
            "evidence": "cup_to_disc_ratio", "value": str(ff["cup_to_disc_ratio"]),
        })
    if ff.get("isnt_rule_followed") is False:
        edges.append({
            "src": rim_id, "rel": "SUPPORTS_DIAGNOSIS", "dst": diag_id,
            "evidence": "isnt_rule_violated", "value": "true",
        })
    active = [s for s in ("bayoneting", "notching", "laminar_dot_sign") if ff.get(s) is True]
    if active:
        edges.append({
            "src": path_id, "rel": "SUPPORTS_DIAGNOSIS", "dst": diag_id,
            "evidence": ",".join(active), "value": "true",
        })
 
    return nodes, edges

# 一堆无聊的处理函数，用来处理biomarker。真无聊嘤嘤嘤

In [10]:
all_nodes, all_edges = [], []
seen_risk: set[str] = set()
 
for _, row in df.iterrows():
    split  = row.get("_split", "")
    record = {k: v for k, v in row.to_dict().items() if k != "_split"}
    nodes, edges = extract_record(record, split=split)
 
    for n in nodes:
        if n["label"] == "RiskLevel":
            if n["id"] in seen_risk:
                continue
            seen_risk.add(n["id"])
        all_nodes.append(n)
    all_edges.extend(edges)
 
nodes_df = pd.DataFrame(all_nodes)
edges_df = pd.DataFrame(all_edges)
 
print(f"Nodes : {len(nodes_df):,}")
print(f"Edges : {len(edges_df):,}")
print("\nNode label breakdown:")
print(nodes_df["label"].value_counts())
print("\nEdge type breakdown:")
print(edges_df["rel"].value_counts())

# 批量处理所有的数据
# 恶心

Nodes : 3,449
Edges : 4,889

Node label breakdown:
label
FundusImage    689
OpticDisc      689
NeuralRim      689
Pathology      689
Diagnosis      689
RiskLevel        4
Name: count, dtype: int64

Edge type breakdown:
rel
SUPPORTS_DIAGNOSIS    1444
HAS_OPTIC_DISC         689
HAS_RIM                689
HAS_PATHOLOGY          689
HAS_DIAGNOSIS          689
HAS_RISK               689
Name: count, dtype: int64


In [11]:
nodes_path = OUT_DIR / "kg_nodes.csv"
edges_path = OUT_DIR / "kg_edges.csv"
 
nodes_df.to_csv(nodes_path, index=False)
edges_df.to_csv(edges_path, index=False)
 
print(f"Saved: {nodes_path}")
print(f"Saved: {edges_path}")

# 以CSV的文件保存

Saved: kg_output/kg_nodes.csv
Saved: kg_output/kg_edges.csv


In [12]:
print("=== FundusImage 样例 ===")
print(nodes_df[nodes_df.label == "FundusImage"].head(3)[
    ["id", "filename", "annotation", "split"]
].to_string(index=False))
 
print("\n=== SUPPORTS_DIAGNOSIS 边样例 ===")
sup = edges_df[edges_df.rel == "SUPPORTS_DIAGNOSIS"]
print(sup.head(5)[["src", "rel", "dst", "evidence", "value"]].to_string(index=False))

# 验证一波

=== FundusImage 样例 ===
          id                filename annotation split
img_351cddb2  acrima_glaucoma_57.png   glaucoma train
img_8b061979  acrima_normal_1058.png     normal train
img_822399c6 acrima_glaucoma_188.png   glaucoma train

=== SUPPORTS_DIAGNOSIS 边样例 ===
          src                rel           dst                             evidence value
  od_14417ee8 SUPPORTS_DIAGNOSIS diag_a5561a59                    cup_to_disc_ratio   0.8
 rim_0e813ca7 SUPPORTS_DIAGNOSIS diag_a5561a59                   isnt_rule_violated  true
path_4f12dd21 SUPPORTS_DIAGNOSIS diag_a5561a59 bayoneting,notching,laminar_dot_sign  true
  od_d314a969 SUPPORTS_DIAGNOSIS diag_dcb19fb3                    cup_to_disc_ratio   0.4
  od_9df47979 SUPPORTS_DIAGNOSIS diag_0b076ea9                    cup_to_disc_ratio   0.7
